# Synthetic data using CEHR-GPT
This uses a Synthea Dataset to train the model and generate synthetic data. The dataset is provided by the authors.

**Important Notes:**

1. Java 8 or 11 required to run pySpark
2. Dataset is provided by authors [here](https://github.com/knatarajan-lab/cehrbert_data/tree/main)
3. Dataset file names start with a '.' so they are hidden files







In [ ]:
import sys
import pyspark
print(pyspark.__version__)
print(sys.version)

In [ ]:
import os, sys

# --- 1. Base Directories ---
# Change this depending on whose environment is running the notebook 
BASE_DIR = ""
EMAIL_ADDRESS = ""
ORNL_USERNAME = ""

# --- 2. OMOP Data Paths ---
OMOP_SYNTHEA_DIR = f"{BASE_DIR}/omop_synthea"
OMOP_SAMPLE_DIR = f"{BASE_DIR}/omop_synthea_sample"

# Sampling Table Variables and Paths
SAMPLE_FRACTION = .01 # 1% sample
SAMPLE_SEED = 42
PERSON_SOURCE = f"{OMOP_SYNTHEA_DIR}/person"
PERSON_TARGET = f"{OMOP_SAMPLE_DIR}/person"
CONCEPT_SOURCE = f"{OMOP_SYNTHEA_DIR}/concept"
CONCEPT_TARGET = f"{OMOP_SAMPLE_DIR}/concept"

# --- 3. Model & Generation Paths ---
CEHRGPT_MODEL_DIR = f"{OMOP_SAMPLE_DIR}/cehrgpt"
SYNTHETIC_DATA_OUT = f"{CEHRGPT_MODEL_DIR}/synthetic_data"

# Example of a separate output path used in your generation step
RESTORED_OMOP_DIR = f"{SYNTHETIC_DATA_OUT}/top_p9500_temp_9000_repetition_penalty_10500/generated_sequences/restored_omop"

# --- 4. Cache Paths ---
HF_CACHE_DIR = f"{BASE_DIR}/.cache"

# --- 5. Apply Environment Variables ---
os.environ.update(
    OMOP_DIR=OMOP_SAMPLE_DIR,
    CEHR_GPT_DATA_DIR=OMOP_SAMPLE_DIR,
    CEHR_GPT_MODEL_DIR=CEHRGPT_MODEL_DIR,
    SYNTHETIC_DATA_OUTPUT_DIR=SYNTHETIC_DATA_OUT,
    HF_HOME=HF_CACHE_DIR,
    HF_DATASETS_CACHE=HF_CACHE_DIR,
    TRANSFORMERS_CACHE=HF_CACHE_DIR
)

# --- 6. Create Directories ---
# Automatically create necessary directories if they do not exist
for d in (OMOP_SYNTHEA_DIR, OMOP_SAMPLE_DIR, CEHRGPT_MODEL_DIR, SYNTHETIC_DATA_OUT):
    os.makedirs(d, exist_ok=True)

print(f"Paths configured successfully | Base Directory: {BASE_DIR}")

### Put these directories where you want to store your OMOP data

In [ ]:
!mkdir omop_synthea
!mkdir omop_synthea_sample
!mkdir omop_synthea_sample/cehrgpt
!mkdir omop_synthea_sample/dataset_prepared
!mkdir omop_synthea_sample/cehrgpt/synthetic_data

In [ ]:
!tar -xvf omop_synthea.tar.gz -C {BASE_DIR}

## Install Java - required for PySpark

In [ ]:
%%bash
export JAVA_HOME=$CONDA_PREFIX
export PATH="$JAVA_HOME/bin:$PATH"
java -version
spark-submit --version

### Set up the PySpark environment varaibles

It is important to configure PySpark environment variables correctly so that Spark knows which Python interpreter to use, how many cores and memory to allocate, and where Spark itself is installed. Without these variables, Spark may default to system settings that are incompatible with your environment, leading to issues such as using the wrong Python version, insufficient memory, or failure to locate Spark binaries. Proper setup ensures consistent execution across driver and executor processes.

### Create the log4j file for pyspark

In [ ]:
import os

# Create the log4j.properties file first
log4j_content = """
log4j.rootLogger=ERROR
log4j.appender.console=org.apache.spark.util.log4j.ConsoleAppender
log4j.appender.console.target=System.err
log4j.appender.console.layout=org.apache.spark.util.log4j.PatternLayout
log4j.appender.console.layout.ConversionPattern=%d{yy/MM/dd HH:mm:ss} %p %c{1}: %m%n

# Silence specific loggers
log4j.logger.org.apache.spark=ERROR
log4j.logger.org.eclipse.jetty=ERROR
log4j.logger.org.apache.hadoop=ERROR
"""

with open('log4j.properties', 'w') as f:
    f.write(log4j_content)

# Update your spark_submit_options to include log4j configuration
log4j_path = os.path.abspath('log4j.properties')

In [ ]:
import os
import subprocess
import pyspark
import cehrbert_data

# Get paths
python_path = subprocess.check_output(['which', 'python']).decode().strip()
spark_home = pyspark.__file__.rsplit('/', 1)[0]
cehrbert_data_home = cehrbert_data.__file__.rsplit('/', 1)[0]


In [ ]:
import os
import subprocess
import pyspark
import cehrbert_data

# Get paths
python_path = subprocess.check_output(['which', 'python']).decode().strip()
spark_home = pyspark.__file__.rsplit('/', 1)[0]
cehrbert_data_home = cehrbert_data.__file__.rsplit('/', 1)[0]

# Set environment variables using magic commands
os.environ['SPARK_HOME'] = spark_home
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
os.environ['SPARK_WORKER_INSTANCES'] = '1'
os.environ['SPARK_WORKER_CORES'] = '16'
os.environ['SPARK_EXECUTOR_CORES'] = '8'
os.environ['SPARK_DRIVER_MEMORY'] = '20g'
os.environ['SPARK_EXECUTOR_MEMORY'] = '20g'
os.environ['SPARK_MASTER'] = 'local[16]'

spark_submit_options = (
    f"--master {os.environ.get('SPARK_MASTER', 'local[*]')} "
    f"--driver-memory {os.environ.get('SPARK_DRIVER_MEMORY', '4g')} "
    f"--executor-memory {os.environ.get('SPARK_EXECUTOR_MEMORY', '4g')} "
    f"--executor-cores {os.environ.get('SPARK_EXECUTOR_CORES', '2')} "
    f"--conf spark.sql.adaptive.enabled={os.environ.get('SPARK_CONF_spark_sql_adaptive_enabled', 'true')} "
    f"--conf spark.sql.adaptive.coalescePartitions.enabled={os.environ.get('SPARK_CONF_spark_sql_adaptive_coalescePartitions_enabled', 'true')} "
    f"--conf spark.serializer={os.environ.get('SPARK_CONF_spark_serializer', 'org.apache.spark.serializer.KryoSerializer')} "
    f"--files log4j.properties "
    f"--conf spark.driver.extraJavaOptions=-Dlog4j.rootLogger=ERROR,console "
    f"--conf spark.executor.extraJavaOptions=-Dlog4j.rootLogger=ERROR,console"
)

# Set the environment variable
os.environ['SPARK_SUBMIT_OPTIONS'] = spark_submit_options

# For paths, you'll still need to use os.environ for concatenation
import os
current_pythonpath = os.environ.get('PYTHONPATH', '')
current_path = os.environ.get('PATH', '')
os.environ['PYTHONPATH'] = f"{spark_home}/python:{current_pythonpath}"
os.environ['PATH'] = f"{spark_home}/bin:{current_path}"

## Create a random OMOP sample

This uses the Synthea OMOP sample data set of 1 million patients provided by the authors here:

[OMOP Sample Dataset](https://drive.google.com/file/d/1k7-cZACaDNw8A1JRI37mfMAhEErxKaQJ/view?usp=share_link)

Based on the person sample, we will extract all the records from the corresponding OMOP tables.
(Note: Skip the sampling step (.sample) step below if using all patients in dataset. File paths in script rely on sample dataset file paths, so your sample would be 100% of data in the sample folders. Run cell below to move to correct folders if not sampling. If sampling, use .sample and files will be in correct folders.).

In [ ]:
import subprocess
import os
 
# Spark code to write to file
spark_code = f"""
from pyspark.sql import SparkSession
 
# Create or get existing spark session
spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
 
# Read, sample, and write
spark.read.parquet('{PERSON_SOURCE}') \\
    .sample(fraction={SAMPLE_FRACTION}, seed={SAMPLE_SEED}) \\
    .write.mode('overwrite') \\
    .parquet('{PERSON_TARGET}')
 
spark.stop()
"""
 
# Write the spark code to a local file
script_path = 'sample_person.py'
with open(script_path, 'w') as f:
    f.write(spark_code)
 
# Use spark-submit instead of pyspark
spark_options = os.environ.get('SPARK_SUBMIT_OPTIONS', '').split()
cmd = ['spark-submit'] + spark_options + [script_path]
 
# Run the command
result = subprocess.run(cmd, capture_output=True, text=True)
print("Return code:", result.returncode)
print("STDOUT:", result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Based on the person sample, we will extract all the records from the corresponding OMOP tables.

In [ ]:
import subprocess
import os
import sys

# Build the command
# CHANGE FILE PATHS @JEREMY
# --person_sample should be the person table from the tar.gz file, and --omop_folder should be the folder containing all the tables from the tar.gz file. The output folder can be a new folder (or same folder)where you want to save the sampled data.
# The first time you run this, --person_sample and --omop_folder should be the same folder.
spark_options = os.environ.get('SPARK_SUBMIT_OPTIONS', '').split()
cmd = ['spark-submit'] + spark_options + [
    f'{cehrbert_data_home}/tools/sample_omop_tables.py',
        '--person_sample', f'{PERSON_SOURCE}',
        '--omop_folder', f'{OMOP_SYNTHEA_DIR}',
    '--output_folder', f'{OMOP_SAMPLE_DIR}'
]

print("Running command:")
print(' '.join(cmd))
print()

# Stream output in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          universal_newlines=True, bufsize=1)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
print(f"\nCommand finished with return code: {return_code}")

In [ ]:
import subprocess
import os
import sys

# Build the command
# --person_sample should be the person table from the tar.gz file, and --omop_folder should be the folder containing all the tables from the tar.gz file. The output folder can be a new folder (or same folder)where you want to save the sampled data.
# The first time you run this, --person_sample and --omop_folder should be the same folder.
spark_options = os.environ.get('SPARK_SUBMIT_OPTIONS', '').split()
cmd = ['spark-submit'] + ["spark-submit subset_omop.py --input f'{OMOP_SYNTHEA_DIR}' --output f'{OMOP_SAMPLE_DIR}' --n 1000"]

print("Running command:")
print(' '.join(cmd))
print()

# Stream output in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          universal_newlines=True, bufsize=1)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
print(f"\nCommand finished with return code: {return_code}")

### Copy concept tables over to where omop tables are from previous cell

In [ ]:
!cp -r f'{OMOP_SYNTHEA_DIR}'/concept /{OMOP_SAMPLE_DIR}
!cp -r f'{OMOP_SYNTHEA_DIR}'/concept_ancestor /{OMOP_SAMPLE_DIR}
!cp -r f'{OMOP_SYNTHEA_DIR}'/concept_relationship /{OMOP_SAMPLE_DIR}

### Set up the environment variables

In [ ]:
os.environ['OMOP_DIR'] = f'{OMOP_SAMPLE_DIR}'
os.environ['CEHR_GPT_DATA_DIR'] = f'{OMOP_SAMPLE_DIR}'
os.environ['CEHR_GPT_MODEL_DIR'] = f'{CEHRGPT_MODEL_DIR}'
os.environ['SYNTHETIC_DATA_OUTPUT_DIR'] = f'{SYNTHETIC_DATA_OUT}'

### Step 1: Generate training data

### Identify Common Concepts
We need to identify the concepts that at least occur more than 100 patient's histories because we don't want to include low-frequency concepts due to privacy concerns.

The current sample data does not contain **measurement** and **observation**, and therefore commented out in the list.

In [ ]:
import subprocess
import os
import sys

# Build the command
spark_options = os.environ.get('SPARK_SUBMIT_OPTIONS', '').split()
cmd = ['spark-submit'] + spark_options + [
    f'{cehrbert_data_home}/apps/generate_included_concept_list.py',
    '-i', os.environ['OMOP_DIR'],
    '-o', os.environ['OMOP_DIR'],
    '--min_num_of_patients', '100',
    '--ehr_table_list',
    "condition_occurrence",
    "procedure_occurrence",
    "drug_exposure",
    #"measurement",
    #"observation",
]

print("Running command:")
print(' '.join(cmd))
print()

# Stream output in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          universal_newlines=True, bufsize=1)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
print(f"\nCommand finished with return code: {return_code}")

### Generate patient sequences for CEHR-GPT
We generate patient sequences excluding the low-frequency concepts. The current sample data does not contain **death**,  **measurement** and **observation**, and therefore commented out in the list.

In [ ]:
import subprocess
import os
import sys

# Build the command
spark_options = os.environ.get('SPARK_SUBMIT_OPTIONS', '').split()
cmd = ['spark-submit'] + spark_options + [
    f'{cehrbert_data_home}/apps/generate_training_data.py',
    '--input_folder', os.environ['OMOP_DIR'],
    '--output_folder', os.environ['CEHR_GPT_DATA_DIR'],
    '-d', '1985-01-01',
    '--att_type', 'day',
    '--inpatient_att_type', 'day',
    '-iv', '-ip',
    '--include_concept_list',
    '--gpt_patient_sequence',
    '--is_new_patient_representation',
    '--include_inpatient_hour_token',
    '--aggregate_by_hour',
    '--should_construct_artificial_visits',
    '--disconnect_problem_list_records',
    #'--include_death',
    '--domain_table_list',
    'condition_occurrence',
    'drug_exposure',
    'procedure_occurrence',
    #"measurement",
    #"observation",
]

print("Running command:")
print(' '.join(cmd))
print()

# Stream output in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          universal_newlines=True, bufsize=1)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
print(f"\nCommand finished with return code: {return_code}")



Let's explore the training data

In [ ]:
import polars as pl
patient_sequences = pl.read_parquet(os.path.join(os.environ['OMOP_DIR'], "patient_sequence/*.parquet"))
patient_sequences

In [ ]:
print(f"Total number of patients: {len(patient_sequences)}")
print(patient_sequences["concept_ids"][299].to_list())

### Step 2: Train CEHR-GPT using generated Patient Sequences

In [ ]:
# --- Script 1: updated_model.sh ---
updated_model_content = f"""#!/bin/bash
#SBATCH -A lrn074                  # Project Account
#SBATCH -J train_model_updated     # Job name
#SBATCH -o %x-%j.out               # Output log
#SBATCH -e %x-%j.err               # Error log
#SBATCH -t 1:00:00                 # Time limit
#SBATCH -N 2                        # Number of Nodes
#SBATCH -c 32                     
#SBATCH -q debug                   
#SBATCH --threads-per-core=1
#SBATCH --mail-user="{EMAIL_ADDRESS}"
#SBATCH --mail-type=ALL


module load PrgEnv-gnu/8.6.0
module load rocm/6.4.0
module load craype-accel-amd-gfx90a
module load miniforge3/23.11.0-0

# --- 1. Load Environment ---
export CONDA_PATH="/autofs/nccs-svm1_sw/frontier/miniforge3/23.11.0-0/condabin/conda"
eval "$($CONDA_PATH shell.bash hook)"
conda activate cehrgpt_env

# --- 2. Set Paths (Injected from Python Configuration) ---
export CEHR_GPT_MODEL_DIR="{CEHRGPT_MODEL_DIR}"
export CEHR_GPT_DATA_DIR="{OMOP_SAMPLE_DIR}"
export OMP_NUM_THREADS=1

# Set the HuggingFace Cache variables
export HF_HOME="{HF_CACHE_DIR}"
export HF_DATASETS_CACHE=$HF_HOME
export TRANSFORMERS_CACHE=$HF_HOME

export TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
export WANDB_DISABLED="true"

export MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
export MASTER_PORT="29500"

# --- Frontier/Cray Network Fabric Variables ---
export FI_CXI_RX_MATCH_MODE=software
export FI_MR_CACHE_MONITOR=memhooks
export NCCL_NET_GDR_LEVEL=3
export NCCL_CROSS_NIC=1
export MIOPEN_USER_DB_PATH="/tmp/${{USER}}-miopen-cache-${{SLURM_JOB_ID}}"
"""

with open("updated_model.sh", "w") as f:
    f.write(updated_model_content)

# --- Script 2: train_model.sh ---
train_model = f"""#!/bin/bash
#SBATCH -A lrn074                  # Project Account
#SBATCH -J train_model     # Job name
#SBATCH -o %x-%j.out               # Output log
#SBATCH -e %x-%j.err               # Error log
#SBATCH -t 1:00:00                 # Time limit
#SBATCH -N 2                        # Number of Nodes
#SBATCH -c 32                     
#SBATCH -q debug                   
#SBATCH --threads-per-core=1
#SBATCH --mail-user="{EMAIL_ADDRESS}"
#SBATCH --mail-type=ALL

module load PrgEnv-gnu/8.6.0
module load rocm/6.4.0
module load craype-accel-amd-gfx90a
module load miniforge3/23.11.0-0

# --- 1. Load Environment ---
export CONDA_PATH="/autofs/nccs-svm1_sw/frontier/miniforge3/23.11.0-0/condabin/conda"
eval "$($CONDA_PATH shell.bash hook)"
conda activate cehrgpt_env

# --- 2. Set Paths & Cache (Injected from Python Configuration) ---
export CEHR_GPT_MODEL_DIR="{CEHRGPT_MODEL_DIR}"
export CEHR_GPT_DATA_DIR="{OMOP_SAMPLE_DIR}"
export OMP_NUM_THREADS=1

# Set the HuggingFace Cache variables
export HF_HOME="{HF_CACHE_DIR}"
export HF_DATASETS_CACHE=$HF_HOME
export TRANSFORMERS_CACHE=$HF_HOME

export TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1

# --- Fix for localhost socket warning ---
export MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
export MASTER_PORT="29500"

# --- Frontier/Cray Network Fabric Variables ---
export FI_CXI_RX_MATCH_MODE=software
export FI_MR_CACHE_MONITOR=memhooks
export NCCL_NET_GDR_LEVEL=3
export NCCL_CROSS_NIC=1
export MIOPEN_USER_DB_PATH="/tmp/${{USER}}-miopen-cache-${{SLURM_JOB_ID}}"

# --- 3. Run the Full Command ---
srun -N2 -n16 --ntasks-per-node=8 -c7 --gpus-per-task=1 --gpu-bind=closest -u python -m cehrgpt.runners.hf_cehrgpt_pretrain_runner \\
    --model_name_or_path "$CEHR_GPT_MODEL_DIR" \\
    --tokenizer_name_or_path "$CEHR_GPT_MODEL_DIR" \\
    --output_dir "$CEHR_GPT_MODEL_DIR" \\
    --data_folder "${{CEHR_GPT_DATA_DIR}}/patient_sequence" \\
    --dataset_prepared_path "${{CEHR_GPT_DATA_DIR}}/dataset_prepared" \\
    --do_train true \\
    --seed 42 \\
    --hidden_size 768 \\
    --num_hidden_layers 12 \\
    --max_position_embeddings 1024 \\
    --sample_packing \\
    --num_train_epochs 10 \\
    --report_to none \\
    --exclude_position_ids false \\
    --max_tokens_per_batch 8192 \\
    --learning_rate 0.0002 \\
    --warmup_ratio 0.02 \\
    --weight_decay 0.01 \\
    --load_best_model_at_end true \\
    --evaluation_strategy epoch \\
    --save_strategy epoch \\
    --bf16 true
"""

with open("train_model.sh", "w") as f:
    f.write(train_model)

print("Created updated_model.sh and train_model.sh successfully!")

In [ ]:
!sbatch train_model.sbatch

In [ ]:
!squeue -u {ORNL_USERNAME}

In [ ]:
!sacct

In [ ]:
!squeue -u {ORNL_USERNAME} --start

In [ ]:
!squeue -p extended

In [ ]:
!scancel -u {ORNL_USERNAME}

### Step 3: Generate synthetic sequences

In [ ]:
# --- Script: generate_sequences.sbatch ---
generate_sequences_content = f"""#!/bin/bash
#SBATCH -A lrn074                  # Project Account
#SBATCH -J cehrgpt_seq_gen         # Job name
#SBATCH -o %x-%j.out               # Output log
#SBATCH -e %x-%j.err               # Error log
#SBATCH -t 8:00:00                 # Time limit (Set to 24h or appropriate limit)
#SBATCH -N 1                       # 1 Node
#SBATCH -c 32                      # 16 CPU cores (Matches dataloader_num_workers)
#SBATCH -p extended                # Partition (verify this with your cluster docs)
#SBATCH --threads-per-core=1
#SBATCH --gres=gpu:8               
#SBATCH --mail-user="{EMAIL_ADDRESS}"
#SBATCH --mail-type=ALL

export CONDA_PATH="/autofs/nccs-svm1_sw/frontier/miniforge3/23.11.0-0/condabin/conda"
eval "$($CONDA_PATH shell.bash hook)"
conda activate cehrgpt_env

# --- Paths Injected from Python Configuration ---
export HF_HOME="{HF_CACHE_DIR}"
export HF_DATASETS_CACHE=$HF_HOME
export TRANSFORMERS_CACHE=$HF_HOME

export TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1

export CEHR_GPT_MODEL_DIR="{CEHRGPT_MODEL_DIR}"
export CEHR_GPT_DATA_DIR="{OMOP_SAMPLE_DIR}"

# Ensure SYNTHETIC_DATA_OUT is correctly defined in your master config block
export SYNTHETIC_DATA_OUTPUT_DIR="{SYNTHETIC_DATA_OUT}"

# 3. Execute the module directly
python -u -m cehrgpt.generation.generate_batch_hf_gpt_sequence \\
    --model_folder "${{CEHR_GPT_MODEL_DIR}}" \\
    --tokenizer_folder "${{CEHR_GPT_MODEL_DIR}}" \\
    --output_folder "${{SYNTHETIC_DATA_OUTPUT_DIR}}" \\
    --num_of_patients 100000 \\
    --batch_size 64 \\
    --buffer_size 1024 \\
    --context_window 1024 \\
    --sampling_strategy TopPStrategy \\
    --top_p 0.95 \\
    --temperature 0.9 \\
    --repetition_penalty 1.05 \\
    --epsilon_cutoff 0.00 \\
    --demographic_data_path "${{CEHR_GPT_DATA_DIR}}/patient_sequence"
"""

# Write to file
with open("generate_sequences.sbatch", "w") as f:
    f.write(generate_sequences_content)

print("Created generate_sequences.sbatch successfully!")

In [ ]:
!sbatch generate_sequences.sbatch

In [ ]:
!squeue -p -extended

In [ ]:
!squeue -u juliannalee --start

Took about 4 hours to generate synthetic sequences for 100,000 patients.

### Step 4: Convert to OMOP Format

In [ ]:
import subprocess
import os
import sys

# Set up the command
cmd = [
    'python', '-m', 'cehrgpt.generation.omop_converter_batch',
    "--patient_sequence_path", "{SYNTHETIC_DATA_OUTPUT_DIR}/top_p9500_temp_9000_repetition_penalty_10500/generated_sequences",
    "--output_folder", "{SYNTHETIC_DATA_OUTPUT_DIR}/top_p9500_temp_9000_repetition_penalty_10500/generated_sequences/restored_omop",
    "--concept_path", "{CEHRGPT_DATA_DIR}/concept",
    '--cpu_cores', "8",
    '--buffer_size', "1024"
]

print("Running command:")
print(' '.join(cmd))
print()

# Stream output in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          universal_newlines=True, bufsize=1)

# Print output line by line as it comes
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

# Wait for process to complete
return_code = process.wait()
print(f"\nPipeline finished with return code: {return_code}")

View OMOP Person Table

In [ ]:
import polars as pl
pl.read_parquet(f"{RESTORED_OMOP_DIR}/person")

### Step 5: Postprocessing for generating random dates

In [ ]:
import pandas as pd

In [ ]:
print(f"Loading person table from {RESTORED_OMOP_DIR}/person ...")
person_df = pd.read_parquet(f"{RESTORED_OMOP_DIR}/person")

print(f"Loading visit_occurrence table from {RESTORED_OMOP_DIR}/visit_occurrence ...")
visit_df = pd.read_parquet(f"{RESTORED_OMOP_DIR}/visit_occurrence")

In [ ]:
def randomize_person_birthdates(person_df, birth_year_col='year_of_birth', month_col='month_of_birth', day_col='day_of_birth'):
    """Randomizes the month and day of birth for each person in the person table while keeping the year of birth unchanged.
    Accounts for the date of the first visit being after the birthdate.
    Args:
        person_df (pd.DataFrame): DataFrame containing the person table.
        birth_year_col (str): Column name for the year of birth.
        month_col (str): Column name for the month of birth.
        day_col (str): Column name for the day of birth.
    Returns:
        pd.DataFrame: Updated person table with randomized month and day of birth."""
    
    n_rows = len(person_df)
    
    person_df[month_col] = np.random.randint(1, 13, size=n_rows)
    
    temp_birth_dates = pd.to_datetime({
        'year': person_df[birth_year_col], 
        'month': person_df[month_col], 
        'day': 1
    })
    
    max_birth_days = temp_birth_dates.dt.daysinmonth 
    person_df[day_col] = np.floor(np.random.random(size=n_rows) * max_birth_days).astype(int) + 1
    
    person_df['birth_date'] = pd.to_datetime({
        'year': person_df[birth_year_col], 
        'month': person_df[month_col], 
        'day': person_df[day_col]
    }).dt.date

    person_df = person_df.drop(columns=[month_col, day_col])
    
    return person_df


def randomize_timeline_by_first_visit(
    visit_df, 
    person_df, 
    person_id_col='person_id', 
    visit_date_col='visit_start_date', 
    visit_end_col='visit_end_date',
    visit_start_dt_col='visit_start_datetime',
    visit_end_dt_col='visit_end_datetime'
):
    """Randomizes the timeline of visits for each person in the visit_occurrence table while keeping the first visit date within the same year.
    Args:
        visit_df (pd.DataFrame): DataFrame containing the visit_occurrence table.
        person_df (pd.DataFrame): DataFrame containing the person table.
        person_id_col (str): Column name for the person ID.
        visit_date_col (str): Column name for the visit start date.
        visit_end_col (str): Column name for the visit end date.
        visit_start_dt_col (str): Column name for the visit start datetime.
        visit_end_dt_col (str): Column name for the visit end datetime.
    Returns:
        pd.DataFrame: Updated visit_occurrence table with randomized visit dates and datetimes."""
    visit_df[visit_date_col] = pd.to_datetime(visit_df[visit_date_col])
    visit_df[visit_end_col] = pd.to_datetime(visit_df[visit_end_col])
    visit_df[visit_start_dt_col] = pd.to_datetime(visit_df[visit_start_dt_col])
    visit_df[visit_end_dt_col] = pd.to_datetime(visit_df[visit_end_dt_col])
    
    first_visits_idx = visit_df.groupby(person_id_col)[visit_date_col].idxmin()
    first_visits_df = visit_df.loc[first_visits_idx].copy()
    
    first_visits_df = first_visits_df.merge(person_df[[person_id_col, 'year_of_birth']], on=person_id_col, how='left')
    
    n_rows = len(first_visits_df)
    
    temp_visit_dates = first_visits_df[visit_date_col]
    visit_year_series = temp_visit_dates.dt.year
    
    visit_year_start = pd.to_datetime({'year': visit_year_series, 'month': 1, 'day': 1})
    visit_year_end = pd.to_datetime({'year': visit_year_series, 'month': 12, 'day': 31})
    
    birth_dates_dt = pd.to_datetime(first_visits_df['year_of_birth'])
    
    min_visit_date = np.maximum(visit_year_start, birth_dates_dt)
    
    valid_days = (visit_year_end - min_visit_date).dt.days
    valid_days = np.maximum(0, valid_days) 
    
    random_days = np.floor(np.random.random(size=n_rows) * (valid_days + 1)).astype(int)
    
    new_first_visit_dates = min_visit_date + pd.to_timedelta(random_days, unit='D')
    
    first_visits_df['time_shift'] = new_first_visit_dates - temp_visit_dates
    
    visit_df = visit_df.merge(first_visits_df[[person_id_col, 'time_shift']], on=person_id_col, how='left')
    
    visit_df[visit_date_col] = visit_df[visit_date_col] + visit_df['time_shift']
    
    visit_df[visit_end_col] = visit_df[visit_end_col] + visit_df['time_shift']
    visit_df.loc[first_visits_idx, visit_end_col] = visit_df.loc[first_visits_idx, visit_date_col]
    
    visit_df[visit_start_dt_col] = visit_df[visit_start_dt_col] + visit_df['time_shift']
    
    visit_df[visit_end_dt_col] = visit_df[visit_end_dt_col] + visit_df['time_shift']
    visit_df.loc[first_visits_idx, visit_end_dt_col] = visit_df.loc[first_visits_idx, visit_start_dt_col]
    
    visit_df[visit_date_col] = visit_df[visit_date_col].dt.date
  
    visit_df[visit_end_col] = visit_df[visit_end_col].dt.date
        
    visit_df = visit_df.drop(columns=['time_shift'])
    
    return visit_df

In [ ]:
date_adj_visit_df = randomize_timeline_by_first_visit(visit_df, person_df, person_id_col='person_id', visit_start_col='visit_start_date', visit_end_col='visit_end_date')
date_adj_person_df = randomize_person_birthdates(person_df, birth_year_col='year_of_birth', month_col='month_of_birth', day_col='day_of_birth')

Read out data to parquet file as needed (need pyarrow)

In [ ]:
date_adj_person_df.to_parquet(f"{RESTORED_OMOP_DIR}/person_randomized_date.parquet", index=False)
date_adj_visit_df.to_parquet(f"{RESTORED_OMOP_DIR}/visit_occurrence_randomized_date.parquet", index=False)